# Supplementary Table 19 — the L2G training set

The complete labelled set the L2G model was trained and evaluated on, positives and negatives
together: `data/l2g_training_set/20250625_gentropy_paper_v1`, **132,970 rows** — 8,520 positive and
124,450 negative. One row per credible-set/gene pair.

The parquet is written by cell 119 of `02_training_set.ipynb` in
`~/Projects/EGL_and_training_set/2506`, and this repository's copy was verified byte-identical to
the one under `~/Projects/EGL_and_training_set/archive/gentropy_paper/data/`. It is the effector
gene list (`ST18`) after the pipeline of that notebook:

1. the L2G feature matrix joined to credible sets and to each study's `diseaseIds`;
2. a pair labelled positive where `array_contains(diseaseIds, EGL.diseaseId)` and the gene matches;
3. restricted to qualified studies and to credible sets with a replicated signal;
4. credible sets with more than two positives dropped;
5. genes within a STRING interaction (score >= 0.8) of a positive gene dropped from the negatives;
6. positives whose gene footprint contains the sentinel dropped, and non-protein-coding genes dropped;
7. de-duplicated on `(geneId, diseaseIds, variantId)` plus the rounded colocalisation features.

Steps 3 and 4 read two GCS-only inputs — `qualified_studies_with_oncology` and
`list_of_gwas_replicated_CSs.parquet` — so the set cannot be rebuilt from this repository. This
notebook labels and exports the saved artefact; it does not reconstruct it.

`test_v3.parquet`, the saved held-out split, is a subset of these rows. The split seed was never
recorded, so that file is the only way to recover which rows were held out; the flag below is read
off it rather than regenerated.

In [1]:
import pandas as pd
import pyarrow.dataset as pads

from manuscript_methods import paper

TRAINING_SET = paper.ROOT / "data" / "l2g_training_set" / "20250625_gentropy_paper_v1"
TEST_SET = paper.ROOT / "data" / "l2g_training_set" / "test_v3.parquet"
SHEET = paper.ROOT / "chapters/06-supplementary-tables/sheets/ST19_l2g_training_set.csv"

gold = pads.dataset(str(TRAINING_SET), format="parquet").to_table().to_pandas()
print(f"rows: {len(gold):,}")
print(gold["goldStandardSet"].value_counts().to_string())
print(f"credible sets: {gold['studyLocusId'].nunique():,} | studies: {gold['studyId'].nunique():,}")
print(f"genes: {gold['geneId'].nunique():,}")

rows: 132,970
goldStandardSet
negative    124450
positive      8520
credible sets: 8,235 | studies: 2,128
genes: 5,197


## Labels and the held-out flag

Gene symbol from the target index and disease names from the disease index of the 25.06 release.
`diseaseIds` is the study's full disease list, so it stays a list: ids are serialised `a;b` and the
names alongside them in the same order, the same convention ST7 uses. A disease id absent from the
index keeps its id in the id column and contributes an empty name.

`Held out` marks the rows of `test_v3.parquet`, matched on `(studyLocusId, geneId)`.

In [2]:
genes = (
    pads.dataset(paper.release("target"), format="parquet")
    .to_table(columns=["id", "approvedSymbol"])
    .to_pandas()
    .rename(columns={"id": "geneId"})
    .drop_duplicates(subset="geneId")
)
disease_name = (
    pads.dataset(paper.release("disease"), format="parquet")
    .to_table(columns=["id", "name"])
    .to_pandas()
    .drop_duplicates(subset="id")
    .set_index("id")["name"]
    .to_dict()
)

labelled = gold.merge(genes, on="geneId", how="left")
assert len(labelled) == len(gold), "the symbol join changed the number of rows"

labelled["diseaseIdList"] = labelled["diseaseIds"].map(lambda ids: ";".join(ids) if ids is not None else "")
labelled["diseaseNameList"] = labelled["diseaseIds"].map(
    lambda ids: ";".join(disease_name.get(i, "") for i in ids) if ids is not None else ""
)

unnamed = {i for ids in labelled["diseaseIds"] if ids is not None for i in ids if i not in disease_name}
print(f"genes with no symbol in the target index: {labelled['approvedSymbol'].isna().sum():,}")
print(f"disease ids with no name in the disease index: {len(unnamed):,}")
print(f"rows carrying more than one disease id: {(labelled['diseaseIds'].map(len) > 1).sum():,}")

genes with no symbol in the target index: 0
disease ids with no name in the disease index: 0
rows carrying more than one disease id: 57,057


In [3]:
test = (
    pads.dataset(str(TEST_SET), format="parquet")
    .to_table(columns=["studyLocusId", "geneId"])
    .to_pandas()
    .drop_duplicates()
)
test["heldOut"] = 1
labelled = labelled.merge(test, on=["studyLocusId", "geneId"], how="left")
assert len(labelled) == len(gold), "the held-out join changed the number of rows"
labelled["heldOut"] = labelled["heldOut"].fillna(0).astype(int)

print(f"held-out rows in test_v3.parquet: {len(test):,}")
print(f"matched into the training set: {int(labelled['heldOut'].sum()):,}")
print(labelled.groupby(["goldStandardSet", "heldOut"]).size().to_string())

held-out rows in test_v3.parquet: 18,611
matched into the training set: 18,611
goldStandardSet  heldOut
negative         0          106973
                 1           17477
positive         0            7386
                 1            1134


In [4]:
st19 = (
    labelled.rename(
        columns={
            "studyLocusId": "Credible set ID",
            "studyId": "Study ID",
            "variantId": "Lead variant ID",
            "geneId": "Ensembl gene ID",
            "approvedSymbol": "Gene symbol",
            "diseaseIdList": "EFO IDs",
            "diseaseNameList": "Disease names",
            "goldStandardSet": "Gold standard set",
            "heldOut": "Held out (1 = test set)",
        }
    )[
        [
            "Credible set ID",
            "Study ID",
            "Lead variant ID",
            "Ensembl gene ID",
            "Gene symbol",
            "EFO IDs",
            "Disease names",
            "Gold standard set",
            "Held out (1 = test set)",
        ]
    ]
    .sort_values(["Credible set ID", "Ensembl gene ID"])
    .reset_index(drop=True)
)
st19.to_csv(SHEET, index=False)

print(f"rows: {len(st19):,}")
print(st19["Gold standard set"].value_counts().to_string())
st19.head()

rows: 132,970
Gold standard set
negative    124450
positive      8520


,Credible set ID,Study ID,Lead variant ID,Ensembl gene ID,Gene symbol,EFO IDs,Disease names,Gold standard set,Held out (1 = test set)
0,00091a12736d53831ee3a4d932bf0834,GCST90244135,6_154039662_A_G,ENSG00000074706,IPCEF1,EFO_0010702,opioid use disorder,negative,1
1,00091a12736d53831ee3a4d932bf0834,GCST90244135,6_154039662_A_G,ENSG00000112038,OPRM1,EFO_0010702,opioid use disorder,positive,1
2,00091a12736d53831ee3a4d932bf0834,GCST90244135,6_154039662_A_G,ENSG00000153721,CNKSR3,EFO_0010702,opioid use disorder,negative,1
3,0010b51a6410b3b0b38e7321c9ae259f,GCST90014007,20_45919050_G_A,ENSG00000062598,ELMO2,EFO_0004612,high density lipoprotein cholesterol measurement,negative,0
4,0010b51a6410b3b0b38e7321c9ae259f,GCST90014007,20_45919050_G_A,ENSG00000064601,CTSA,EFO_0004612,high density lipoprotein cholesterol measurement,negative,0


## Against the effector gene list

`ST18` is the input, this sheet is the output. The positives here are the EGL pairs that survived
being matched to a credible set of a qualified, replicated study; the check is at the row level,
because a positive is flagged by `array_contains` and a multi-disease study contributes one matched
pair alongside several co-occurring ones.

In [5]:
egl = (
    pads.dataset(paper.baseline("EGL.parquet"), format="parquet")
    .to_table(columns=["diseaseId", "targetId"])
    .to_pandas()
)
pairs = set(map(tuple, egl[["targetId", "diseaseId"]].to_numpy()))

positives = gold[gold["goldStandardSet"] == "positive"]
matched = [
    any((gene, disease) in pairs for disease in diseases)
    for gene, diseases in zip(positives["geneId"], positives["diseaseIds"])
]
assert all(matched), "a positive row has no disease id in the effector gene list"

distinct_positive_pairs = (
    positives.explode("diseaseIds")
    .rename(columns={"diseaseIds": "diseaseId", "geneId": "targetId"})[["diseaseId", "targetId"]]
    .drop_duplicates()
    .merge(egl, on=["diseaseId", "targetId"], how="inner")
)
print(f"EGL pairs: {len(egl):,}")
print(f"positive rows: {len(positives):,}, all matched to the EGL")
print(f"EGL pairs behind them: {len(distinct_positive_pairs):,}")

paper.save_results(
    "supplementary_table_l2g_training_set",
    {
        "rows": len(st19),
        "positive_rows": int((st19["Gold standard set"] == "positive").sum()),
        "negative_rows": int((st19["Gold standard set"] == "negative").sum()),
        "credible_sets": int(gold["studyLocusId"].nunique()),
        "genes": int(gold["geneId"].nunique()),
        "held_out_rows": int(st19["Held out (1 = test set)"].sum()),
        "egl_pairs_behind_the_positives": len(distinct_positive_pairs),
    },
)

EGL pairs: 42,288
positive rows: 8,520, all matched to the EGL
EGL pairs behind them: 612


'/Users/yt4/Projects/Gentropy-manuscript/results/supplementary_table_l2g_training_set.json'